# 从零实现 DPO 偏好优化：序列 log-prob、参考策略与泄漏防线

本 Notebook 手写 TinyAutoregressivePolicy56、MaskedSequenceLogProb56 和 DPOObjective56，不调用 transformers、TRL、PEFT 或现成训练器。我们从 chosen/rejected pair 的数据合同开始，实现 padding-aware 序列分数、冻结 reference、DPO log-ratio、label smoothing、受控训练和可信发布。

微型任务只学习两类提示对 CLEAR/CAUTIOUS 的固定偏好。它适合验证符号、mask 和梯度，不足以证明真实对齐、安全性或人类偏好泛化。

In [ ]:
import copy  # 导入本单元所需的依赖。
import hashlib  # 导入本单元所需的依赖。
import json  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
import random  # 导入本单元所需的依赖。
import warnings  # 导入本单元所需的依赖。
from types import MappingProxyType  # 导入本单元所需的依赖。

warnings.filterwarnings("ignore", message="The pynvml package is deprecated")  # 计算并保存当前步骤的中间状态。
import numpy as np  # 导入本单元所需的依赖。
import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。

SEED56 = 5601  # 计算并保存当前步骤的中间状态。
random.seed(SEED56); np.random.seed(SEED56); torch.manual_seed(SEED56)  # 执行当前语句以推进本节示例。
torch.set_num_threads(1)  # 执行当前语句以推进本节示例。
DEVICE56 = torch.device("cpu")  # 计算并保存当前步骤的中间状态。

def canonical_json56(value):  # 定义本节可复用的核心函数。
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))  # 返回当前分支计算出的结果。

def sha56(raw):  # 定义本节可复用的核心函数。
    return hashlib.sha256(raw).hexdigest()  # 返回当前分支计算出的结果。

assert DEVICE56.type == "cpu"  # 用受控断言验证关键不变量。
assert torch.get_num_threads() == 1  # 用受控断言验证关键不变量。

## 1. Preference pair、有效上下文指纹与三类切分

TinyAutoregressivePolicy56 是一阶模型：首个 response token 只看到 prompt 最后一个 task token，较早的唯一 USER token 不在有效上下文中。若按完整 prompt 去重，USER id 会掩盖同一偏好规则跨 split 重复。因此 effective_fingerprint56 绑定模型真正消费的 transition contexts、chosen/rejected targets、EOS 与模板版本，并忽略无效的 USER 前缀。

train 含 FACT/STYLE 两条规则；known_rule_regression 用新 USER 复测相同规则，明确允许与 train 的有效指纹重叠；true_holdout 使用训练从未见过的 TASK_HOLDOUT。三者分别报告，不能把 known-rule regression 写成独立泛化。

In [ ]:
TOKENIZER56 = {  # 计算并保存当前步骤的中间状态。
    "<pad>": 0, "<bos>": 1, "<eos>": 2,  # 执行当前语句以推进本节示例。
    "<task_fact>": 3, "<task_style>": 4,  # 执行当前语句以推进本节示例。
    "<user0>": 5, "<user1>": 6, "<user2>": 7,  # 执行当前语句以推进本节示例。
    "<user3>": 8, "<user4>": 9, "<user5>": 10,  # 执行当前语句以推进本节示例。
    "CLEAR": 11, "CAUTIOUS": 12,  # 执行当前语句以推进本节示例。
    "<user6>": 13, "<task_holdout>": 14,  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
DATA56 = [  # 计算并保存当前步骤的中间状态。
    {"id": "p0", "prompt": [1, 5, 3], "chosen": [11, 2], "rejected": [12, 2]},  # 执行当前语句以推进本节示例。
    {"id": "p1", "prompt": [1, 6, 4], "chosen": [12, 2], "rejected": [11, 2]},  # 执行当前语句以推进本节示例。
    {"id": "p4", "prompt": [1, 9, 3], "chosen": [11, 2], "rejected": [12, 2]},  # 执行当前语句以推进本节示例。
    {"id": "p5", "prompt": [1, 10, 4], "chosen": [12, 2], "rejected": [11, 2]},  # 执行当前语句以推进本节示例。
    {"id": "p6", "prompt": [1, 13, 14], "chosen": [11, 2], "rejected": [12, 2]},  # 执行当前语句以推进本节示例。
]  # 执行当前语句以推进本节示例。
SPLIT56 = {  # 计算并保存当前步骤的中间状态。
    "train": ["p0", "p1"],  # 执行当前语句以推进本节示例。
    "known_rule_regression": ["p4", "p5"],  # 执行当前语句以推进本节示例。
    "true_holdout": ["p6"],  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。

def effective_fingerprint56(row):  # 定义本节可复用的核心函数。
    def transitions(response):  # 定义本节可复用的核心函数。
        return {  # 返回当前分支计算出的结果。
            "contexts": [row["prompt"][-1]] + response[:-1],  # 执行当前语句以推进本节示例。
            "targets": list(response),  # 执行当前语句以推进本节示例。
        }  # 执行当前语句以推进本节示例。
    return canonical_json56({  # 返回当前分支计算出的结果。
        "template": "bigram-prompt-response-eos-v1",  # 执行当前语句以推进本节示例。
        "chosen": transitions(row["chosen"]),  # 执行当前语句以推进本节示例。
        "rejected": transitions(row["rejected"]),  # 执行当前语句以推进本节示例。
    })  # 执行当前语句以推进本节示例。

def validate_preference_data56(records, split, tokenizer):  # 定义本节可复用的核心函数。
    ids = [row["id"] for row in records]  # 计算并保存当前步骤的中间状态。
    referenced = sum(split.values(), [])  # 计算并保存当前步骤的中间状态。
    if len(ids) != len(set(ids)) or sorted(ids) != sorted(referenced) or len(referenced) != len(set(referenced)):  # 按当前条件选择后续控制路径。
        raise ValueError("record_split_membership_invalid")  # 遇到非法合同立即显式失败。
    by_id = {row["id"]: row for row in records}  # 计算并保存当前步骤的中间状态。
    all_tokens = set(tokenizer.values())  # 计算并保存当前步骤的中间状态。
    seen_prompt, seen_pair = {}, {}  # 计算并保存当前步骤的中间状态。
    for split_name, split_ids in split.items():  # 遍历输入元素以累积或检查结果。
        for record_id in split_ids:  # 遍历输入元素以累积或检查结果。
            row = by_id[record_id]  # 计算并保存当前步骤的中间状态。
            if row["prompt"][0] != tokenizer["<bos>"] or row["chosen"][-1] != tokenizer["<eos>"] or row["rejected"][-1] != tokenizer["<eos>"]:  # 按当前条件选择后续控制路径。
                raise ValueError("prompt_response_boundary_invalid")  # 遇到非法合同立即显式失败。
            if row["chosen"] == row["rejected"]:  # 按当前条件选择后续控制路径。
                raise ValueError("chosen_equals_rejected")  # 遇到非法合同立即显式失败。
            if any(token not in all_tokens for token in row["prompt"] + row["chosen"] + row["rejected"]):  # 按当前条件选择后续控制路径。
                raise ValueError("token_out_of_vocabulary")  # 遇到非法合同立即显式失败。
            prompt_key = tuple(row["prompt"])  # 计算并保存当前步骤的中间状态。
            pair_key = (prompt_key, tuple(sorted((tuple(row["chosen"]), tuple(row["rejected"])))))  # 计算并保存当前步骤的中间状态。
            if prompt_key in seen_prompt:  # 按当前条件选择后续控制路径。
                raise ValueError("prompt_cross_split_leakage" if seen_prompt[prompt_key] != split_name else "duplicate_prompt_in_split")  # 遇到非法合同立即显式失败。
            if pair_key in seen_pair:  # 按当前条件选择后续控制路径。
                raise ValueError("pair_cross_split_leakage" if seen_pair[pair_key] != split_name else "duplicate_pair_in_split")  # 遇到非法合同立即显式失败。
            seen_prompt[prompt_key], seen_pair[pair_key] = split_name, split_name  # 计算并保存当前步骤的中间状态。
    return True  # 返回当前分支计算出的结果。

assert validate_preference_data56(DATA56, SPLIT56, TOKENIZER56)  # 用受控断言验证关键不变量。
assert len(set(TOKENIZER56.values())) == len(TOKENIZER56)  # 用受控断言验证关键不变量。
assert not (set(SPLIT56["train"]) & set(SPLIT56["true_holdout"]))  # 用受控断言验证关键不变量。
table_for_fingerprint56 = {row["id"]: row for row in DATA56}  # 计算并保存当前步骤的中间状态。
fingerprints56 = {  # 计算并保存当前步骤的中间状态。
    name: [effective_fingerprint56(table_for_fingerprint56[record_id]) for record_id in ids]  # 执行当前语句以推进本节示例。
    for name, ids in SPLIT56.items()  # 遍历输入元素以累积或检查结果。
}  # 执行当前语句以推进本节示例。
assert all(len(values) == len(set(values)) for values in fingerprints56.values())  # 用受控断言验证关键不变量。
assert set(fingerprints56["known_rule_regression"]) == set(fingerprints56["train"])  # 用受控断言验证关键不变量。
assert set(fingerprints56["true_holdout"]).isdisjoint(fingerprints56["train"])  # 用受控断言验证关键不变量。
assert effective_fingerprint56(table_for_fingerprint56["p0"]) == effective_fingerprint56(table_for_fingerprint56["p4"])  # 用受控断言验证关键不变量。
assert effective_fingerprint56(table_for_fingerprint56["p1"]) == effective_fingerprint56(table_for_fingerprint56["p5"])  # 用受控断言验证关键不变量。

## 2. 自回归策略与 masked sequence log-probability

TinyAutoregressivePolicy56 是一阶因果模型：位置 t 的 logits 只依赖 token_t，用来预测 token_{t+1}。虽然容量远小于 Transformer，它仍是合法的自回归分解，且足以隔离 DPO 算法本身。

对 response mask m_t，序列分数为 sum_t m_t log π(y_t | x,y_<t)。padding、prompt-only 位置都不得进入分数。length_mode 可选 sum 或 mean；两者优化目标不同，必须写入 recipe，不能训练用 sum、线上比较却偷偷改成 mean。

In [ ]:
class TinyAutoregressivePolicy56(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, vocab_size=15, dim=20):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.vocab_size, self.dim = vocab_size, dim  # 计算并保存当前步骤的中间状态。
        self.embedding = nn.Embedding(vocab_size, dim)  # 计算并保存当前步骤的中间状态。
        self.hidden = nn.Linear(dim, dim)  # 计算并保存当前步骤的中间状态。
        self.head = nn.Linear(dim, vocab_size)  # 计算并保存当前步骤的中间状态。

    def forward(self, input_ids, attention_mask):  # 定义本节可复用的核心函数。
        if input_ids.ndim != 2 or input_ids.dtype != torch.long or attention_mask.shape != input_ids.shape:  # 按当前条件选择后续控制路径。
            raise ValueError("invalid_policy_inputs")  # 遇到非法合同立即显式失败。
        if attention_mask.dtype != torch.bool:  # 按当前条件选择后续控制路径。
            raise ValueError("attention_mask_must_be_bool")  # 遇到非法合同立即显式失败。
        if input_ids.min() < 0 or input_ids.max() >= self.vocab_size:  # 按当前条件选择后续控制路径。
            raise ValueError("token_id_out_of_range")  # 遇到非法合同立即显式失败。
        hidden = torch.tanh(self.hidden(self.embedding(input_ids)))  # 计算并保存当前步骤的中间状态。
        logits = self.head(hidden)  # 计算并保存当前步骤的中间状态。
        return logits * attention_mask.unsqueeze(-1)  # 返回当前分支计算出的结果。

class MaskedSequenceLogProb56(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, length_mode="sum"):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if length_mode not in ("sum", "mean"):  # 按当前条件选择后续控制路径。
            raise ValueError("length_mode_must_be_sum_or_mean")  # 遇到非法合同立即显式失败。
        self.length_mode = length_mode  # 计算并保存当前步骤的中间状态。

    def forward(self, model, input_ids, attention_mask, response_mask):  # 定义本节可复用的核心函数。
        if response_mask.shape != (input_ids.shape[0], input_ids.shape[1] - 1) or response_mask.dtype != torch.bool:  # 按当前条件选择后续控制路径。
            raise ValueError("invalid_response_mask")  # 遇到非法合同立即显式失败。
        logits = model(input_ids[:, :-1], attention_mask[:, :-1])  # 计算并保存当前步骤的中间状态。
        targets = input_ids[:, 1:]  # 计算并保存当前步骤的中间状态。
        valid = response_mask & attention_mask[:, :-1] & attention_mask[:, 1:]  # 计算并保存当前步骤的中间状态。
        if not valid.any(dim=1).all():  # 按当前条件选择后续控制路径。
            raise ValueError("each_sequence_needs_scored_response_token")  # 遇到非法合同立即显式失败。
        token_logp = logits.log_softmax(-1).gather(-1, targets.unsqueeze(-1)).squeeze(-1)  # 计算并保存当前步骤的中间状态。
        total = (token_logp * valid).sum(-1)  # 计算并保存当前步骤的中间状态。
        if self.length_mode == "mean":  # 按当前条件选择后续控制路径。
            total = total / valid.sum(-1)  # 计算并保存当前步骤的中间状态。
        return total  # 返回当前分支计算出的结果。

shape_policy56 = TinyAutoregressivePolicy56()  # 计算并保存当前步骤的中间状态。
shape_ids56 = torch.tensor([[1, 5, 3, 11, 2]])  # 计算并保存当前步骤的中间状态。
shape_attention56 = torch.ones_like(shape_ids56, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
shape_response56 = torch.tensor([[False, False, True, True]])  # 计算并保存当前步骤的中间状态。
shape_score56 = MaskedSequenceLogProb56("sum")(shape_policy56, shape_ids56, shape_attention56, shape_response56)  # 计算并保存当前步骤的中间状态。
assert shape_policy56(shape_ids56, shape_attention56).shape == (1, 5, 15)  # 用受控断言验证关键不变量。
assert shape_score56.shape == (1,)  # 用受控断言验证关键不变量。
assert torch.isfinite(shape_score56).all()  # 用受控断言验证关键不变量。

## 3. Collate：prompt 不计分，response 与 EOS 计分

序列为 prompt + response，模型输入长度 T，next-token 分数长度 T-1。第一个 response token 对应 mask 下标 prompt_len-1；末尾 EOS 也计入偏好概率。collate 对 chosen/rejected 独立 padding，因此 scorer 不能依赖两边恰好同长。

In [ ]:
def collate_side56(records, side):  # 定义本节可复用的核心函数。
    sequences = [row["prompt"] + row[side] for row in records]  # 计算并保存当前步骤的中间状态。
    widths = max(map(len, sequences))  # 计算并保存当前步骤的中间状态。
    ids = torch.full((len(records), widths), TOKENIZER56["<pad>"], dtype=torch.long)  # 计算并保存当前步骤的中间状态。
    attention = torch.zeros((len(records), widths), dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
    response = torch.zeros((len(records), widths - 1), dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
    for index, (row, sequence) in enumerate(zip(records, sequences)):  # 遍历输入元素以累积或检查结果。
        ids[index, :len(sequence)] = torch.tensor(sequence)  # 计算并保存当前步骤的中间状态。
        attention[index, :len(sequence)] = True  # 计算并保存当前步骤的中间状态。
        start = len(row["prompt"]) - 1  # 计算并保存当前步骤的中间状态。
        response[index, start:len(sequence) - 1] = True  # 计算并保存当前步骤的中间状态。
    return ids, attention, response  # 返回当前分支计算出的结果。

table56 = {row["id"]: row for row in DATA56}  # 计算并保存当前步骤的中间状态。
train_records56 = [table56[record_id] for record_id in SPLIT56["train"]]  # 计算并保存当前步骤的中间状态。
chosen_batch56 = collate_side56(train_records56, "chosen")  # 计算并保存当前步骤的中间状态。
rejected_batch56 = collate_side56(train_records56, "rejected")  # 计算并保存当前步骤的中间状态。
assert chosen_batch56[0].shape == rejected_batch56[0].shape == (2, 5)  # 用受控断言验证关键不变量。
assert chosen_batch56[2].sum(-1).tolist() == [2, 2]  # 用受控断言验证关键不变量。

scorer56 = MaskedSequenceLogProb56("sum")  # 计算并保存当前步骤的中间状态。
original56 = scorer56(shape_policy56, *chosen_batch56)  # 计算并保存当前步骤的中间状态。
padded_ids56 = F.pad(chosen_batch56[0], (0, 2), value=TOKENIZER56["CAUTIOUS"])  # 计算并保存当前步骤的中间状态。
padded_attention56 = F.pad(chosen_batch56[1], (0, 2), value=False)  # 计算并保存当前步骤的中间状态。
padded_response56 = F.pad(chosen_batch56[2], (0, 2), value=False)  # 计算并保存当前步骤的中间状态。
padded56 = scorer56(shape_policy56, padded_ids56, padded_attention56, padded_response56)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(original56, padded56, atol=0, rtol=0)  # 用受控断言验证关键不变量。
assert padded_response56.shape[1] == padded_ids56.shape[1] - 1  # 用受控断言验证关键不变量。
mean_scores56 = MaskedSequenceLogProb56("mean")(shape_policy56, *chosen_batch56)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(mean_scores56, original56 / chosen_batch56[2].sum(-1), atol=1e-7)  # 用受控断言验证关键不变量。
assert not torch.allclose(mean_scores56, original56)  # 用受控断言验证关键不变量。

unequal_record56 = [{  # 计算并保存当前步骤的中间状态。
    "prompt": [1, 5, 3],  # 执行当前语句以推进本节示例。
    "chosen": [11, 2],  # 执行当前语句以推进本节示例。
    "rejected": [12, 12, 2],  # 执行当前语句以推进本节示例。
}]  # 执行当前语句以推进本节示例。
unequal_chosen56 = collate_side56(unequal_record56, "chosen")  # 计算并保存当前步骤的中间状态。
unequal_rejected56 = collate_side56(unequal_record56, "rejected")  # 计算并保存当前步骤的中间状态。
assert unequal_chosen56[0].shape == (1, 5) and unequal_rejected56[0].shape == (1, 6)  # 用受控断言验证关键不变量。
assert unequal_chosen56[2].sum().item() == 2 and unequal_rejected56[2].sum().item() == 3  # 用受控断言验证关键不变量。
for ids56, attention56, response56 in (unequal_chosen56, unequal_rejected56):  # 遍历输入元素以累积或检查结果。
    last56 = int(attention56[0].sum()) - 1  # 计算并保存当前步骤的中间状态。
    assert ids56[0, last56].item() == TOKENIZER56["<eos>"]  # 用受控断言验证关键不变量。
    assert response56[0, last56 - 1]  # 用受控断言验证关键不变量。

hand_ids56, hand_attention56, hand_response56 = unequal_chosen56  # 计算并保存当前步骤的中间状态。
hand_logits56 = shape_policy56(hand_ids56[:, :-1], hand_attention56[:, :-1]).log_softmax(-1)  # 计算并保存当前步骤的中间状态。
hand_targets56 = hand_ids56[:, 1:]  # 计算并保存当前步骤的中间状态。
hand_token_logp56 = hand_logits56.gather(-1, hand_targets56.unsqueeze(-1)).squeeze(-1)  # 计算并保存当前步骤的中间状态。
hand_expected56 = hand_token_logp56[hand_response56].sum()  # 计算并保存当前步骤的中间状态。
hand_actual56 = scorer56(shape_policy56, *unequal_chosen56)[0]  # 计算并保存当前步骤的中间状态。
assert torch.allclose(hand_actual56, hand_expected56, atol=1e-7)  # 用受控断言验证关键不变量。
changed_user_ids56 = hand_ids56.clone()  # 计算并保存当前步骤的中间状态。
changed_user_ids56[0, 1] = TOKENIZER56["<user3>"]  # 计算并保存当前步骤的中间状态。
changed_user56 = scorer56(shape_policy56, changed_user_ids56, hand_attention56, hand_response56)[0]  # 计算并保存当前步骤的中间状态。
assert torch.allclose(hand_actual56, changed_user56, atol=0, rtol=0)  # 用受控断言验证关键不变量。

## 4. DPO log-ratio、β 与 label smoothing

记 d_policy=logπ(chosen)-logπ(rejected)，d_ref 同理，z=β(d_policy-d_ref)。无平滑 DPO loss 是 -log σ(z)。加入 ε 后使用

-(1-ε) log σ(z) - ε log σ(-z)。

reference=policy 时 z=0，单样本 loss 必须为 log 2。交换 chosen/rejected 会令 z 变号。实现用 logsigmoid，避免先算 sigmoid 再 log 导致极端 logit 下溢。

In [ ]:
class DPOObjective56(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, beta=0.2, label_smoothing=0.0):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if not beta > 0 or not 0 <= label_smoothing < 0.5:  # 按当前条件选择后续控制路径。
            raise ValueError("invalid_beta_or_label_smoothing")  # 遇到非法合同立即显式失败。
        self.beta = float(beta)  # 计算并保存当前步骤的中间状态。
        self.label_smoothing = float(label_smoothing)  # 计算并保存当前步骤的中间状态。

    def forward(self, policy_chosen, policy_rejected, reference_chosen, reference_rejected):  # 定义本节可复用的核心函数。
        shapes = {tuple(value.shape) for value in (policy_chosen, policy_rejected, reference_chosen, reference_rejected)}  # 计算并保存当前步骤的中间状态。
        if len(shapes) != 1:  # 按当前条件选择后续控制路径。
            raise ValueError("logprob_shapes_must_match")  # 遇到非法合同立即显式失败。
        log_ratio = (policy_chosen - policy_rejected) - (reference_chosen - reference_rejected)  # 计算并保存当前步骤的中间状态。
        z = self.beta * log_ratio  # 计算并保存当前步骤的中间状态。
        eps = self.label_smoothing  # 计算并保存当前步骤的中间状态。
        loss = -(1 - eps) * F.logsigmoid(z) - eps * F.logsigmoid(-z)  # 计算并保存当前步骤的中间状态。
        return loss.mean(), log_ratio  # 返回当前分支计算出的结果。

objective_probe56 = DPOObjective56(beta=0.3, label_smoothing=0.1)  # 计算并保存当前步骤的中间状态。
equal56 = torch.tensor([2.0, -4.0])  # 计算并保存当前步骤的中间状态。
baseline_loss56, baseline_ratio56 = objective_probe56(equal56, -equal56, equal56, -equal56)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(baseline_ratio56, torch.zeros(2))  # 用受控断言验证关键不变量。
assert abs(float(baseline_loss56) - math.log(2)) < 1e-6  # 用受控断言验证关键不变量。

positive_loss56, positive_ratio56 = objective_probe56(torch.tensor([3.0]), torch.tensor([0.0]), torch.tensor([1.0]), torch.tensor([0.0]))  # 计算并保存当前步骤的中间状态。
flipped_loss56, flipped_ratio56 = objective_probe56(torch.tensor([0.0]), torch.tensor([3.0]), torch.tensor([0.0]), torch.tensor([1.0]))  # 计算并保存当前步骤的中间状态。
assert positive_ratio56.item() == -flipped_ratio56.item() == 2.0  # 用受控断言验证关键不变量。
assert positive_loss56 < flipped_loss56  # 用受控断言验证关键不变量。
extreme_loss56, extreme_ratio56 = objective_probe56(torch.tensor([1e6, -1e6]), torch.tensor([-1e6, 1e6]), torch.zeros(2), torch.zeros(2))  # 计算并保存当前步骤的中间状态。
assert torch.isfinite(extreme_loss56)  # 用受控断言验证关键不变量。
assert torch.isfinite(extreme_ratio56).all()  # 用受控断言验证关键不变量。

## 5. 严格冻结 reference 并执行受控训练

policy 与 reference 从同一初始 state 开始；reference 参数 requires_grad=False 且保持 eval。reference log-prob 在 no_grad 区域计算。只有 policy 进入 optimizer。

这里使用 sum 长度语义、β=0.3、ε=0.05。由于 chosen/rejected 等长，实验没有研究长度偏好；真实数据若长度不同，需要报告 sum/mean、EOS、截断和最大长度策略的敏感性。

In [ ]:
reference56 = TinyAutoregressivePolicy56()  # 计算并保存当前步骤的中间状态。
policy56 = copy.deepcopy(reference56)  # 计算并保存当前步骤的中间状态。
reference56.eval()  # 执行当前语句以推进本节示例。
for parameter56 in reference56.parameters():  # 遍历输入元素以累积或检查结果。
    parameter56.requires_grad_(False)  # 执行当前语句以推进本节示例。
assert not any(p.requires_grad for p in reference56.parameters())  # 用受控断言验证关键不变量。
reference_state_before56 = {key: value.detach().cpu().clone() for key, value in reference56.state_dict().items()}  # 计算并保存当前步骤的中间状态。
def frozen_state_digest56(state):  # 定义本节可复用的核心函数。
    digest56 = hashlib.sha256()  # 计算并保存当前步骤的中间状态。
    for key56 in sorted(state):  # 遍历输入元素以累积或检查结果。
        tensor56 = state[key56].detach().cpu().contiguous()  # 计算并保存当前步骤的中间状态。
        header56 = canonical_json56({"key": key56, "dtype": str(tensor56.dtype), "shape": list(tensor56.shape)}).encode()  # 计算并保存当前步骤的中间状态。
        raw56 = tensor56.numpy().tobytes()  # 计算并保存当前步骤的中间状态。
        digest56.update(len(header56).to_bytes(8, "big")); digest56.update(header56)  # 执行当前语句以推进本节示例。
        digest56.update(len(raw56).to_bytes(8, "big")); digest56.update(raw56)  # 执行当前语句以推进本节示例。
    return digest56.hexdigest()  # 返回当前分支计算出的结果。
reference_digest_before56 = frozen_state_digest56(reference_state_before56)  # 计算并保存当前步骤的中间状态。

scorer56 = MaskedSequenceLogProb56("sum")  # 计算并保存当前步骤的中间状态。
objective56 = DPOObjective56(beta=0.3, label_smoothing=0.05)  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    ref_chosen56 = scorer56(reference56, *chosen_batch56)  # 计算并保存当前步骤的中间状态。
    ref_rejected56 = scorer56(reference56, *rejected_batch56)  # 计算并保存当前步骤的中间状态。
initial_chosen56 = scorer56(policy56, *chosen_batch56)  # 计算并保存当前步骤的中间状态。
initial_rejected56 = scorer56(policy56, *rejected_batch56)  # 计算并保存当前步骤的中间状态。
initial_loss56, initial_ratio56 = objective56(initial_chosen56, initial_rejected56, ref_chosen56, ref_rejected56)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(initial_ratio56, torch.zeros_like(initial_ratio56), atol=1e-7)  # 用受控断言验证关键不变量。

optimizer56 = torch.optim.AdamW(policy56.parameters(), lr=0.045, weight_decay=0.0)  # 计算并保存当前步骤的中间状态。
for _ in range(85):  # 遍历输入元素以累积或检查结果。
    optimizer56.zero_grad()  # 执行当前语句以推进本节示例。
    policy_chosen56 = scorer56(policy56, *chosen_batch56)  # 计算并保存当前步骤的中间状态。
    policy_rejected56 = scorer56(policy56, *rejected_batch56)  # 计算并保存当前步骤的中间状态。
    loss56, ratio56 = objective56(policy_chosen56, policy_rejected56, ref_chosen56, ref_rejected56)  # 计算并保存当前步骤的中间状态。
    loss56.backward(); optimizer56.step()  # 执行当前语句以推进本节示例。

final_chosen56 = scorer56(policy56, *chosen_batch56)  # 计算并保存当前步骤的中间状态。
final_rejected56 = scorer56(policy56, *rejected_batch56)  # 计算并保存当前步骤的中间状态。
final_loss56, final_ratio56 = objective56(final_chosen56, final_rejected56, ref_chosen56, ref_rejected56)  # 计算并保存当前步骤的中间状态。
assert final_loss56 < initial_loss56 * 0.35  # 用受控断言验证关键不变量。
assert (final_chosen56 > final_rejected56).all()  # 用受控断言验证关键不变量。
assert (final_ratio56 > 0).all()  # 用受控断言验证关键不变量。
assert all(p.grad is None for p in reference56.parameters())  # 用受控断言验证关键不变量。
reference_state_after56 = {key: value.detach().cpu().clone() for key, value in reference56.state_dict().items()}  # 计算并保存当前步骤的中间状态。
assert reference_digest_before56 == frozen_state_digest56(reference_state_after56)  # 用受控断言验证关键不变量。
assert reference_state_before56.keys() == reference_state_after56.keys()  # 用受控断言验证关键不变量。
assert all(torch.equal(reference_state_before56[key], reference_state_after56[key]) for key in reference_state_before56)  # 用受控断言验证关键不变量。

## 6. known-rule regression 与真正 holdout 分开报告

known_rule_regression 更换了 USER token，但模型的有效 task context 与 train 完全相同，所以高准确率只证明规则回归。true_holdout 的 TASK_HOLDOUT 从未进入训练，不对其准确率设成功门槛，只要求打分有限并明确报告。真实泛化还需未见模板、主题和响应分布的独立集。

In [ ]:
@torch.no_grad()  # 为下方定义附加声明式配置。
def preference_accuracy56(model, record_ids):  # 定义本节可复用的核心函数。
    records = [table56[record_id] for record_id in record_ids]  # 计算并保存当前步骤的中间状态。
    chosen = scorer56(model, *collate_side56(records, "chosen"))  # 计算并保存当前步骤的中间状态。
    rejected = scorer56(model, *collate_side56(records, "rejected"))  # 计算并保存当前步骤的中间状态。
    return float((chosen > rejected).float().mean()), (chosen - rejected)  # 返回当前分支计算出的结果。

train_accuracy56, train_margin56 = preference_accuracy56(policy56, SPLIT56["train"])  # 计算并保存当前步骤的中间状态。
known_accuracy56, known_margin56 = preference_accuracy56(policy56, SPLIT56["known_rule_regression"])  # 计算并保存当前步骤的中间状态。
holdout_accuracy56, holdout_margin56 = preference_accuracy56(policy56, SPLIT56["true_holdout"])  # 计算并保存当前步骤的中间状态。
assert train_accuracy56 == 1.0  # 用受控断言验证关键不变量。
assert known_accuracy56 == 1.0  # 用受控断言验证关键不变量。
assert train_margin56.min() > 1.0 and known_margin56.min() > 1.0  # 用受控断言验证关键不变量。
assert holdout_accuracy56 in (0.0, 1.0) and holdout_margin56.numel() == 1  # 用受控断言验证关键不变量。
assert torch.isfinite(holdout_margin56).all()  # 用受控断言验证关键不变量。

contaminated56 = copy.deepcopy(DATA56)  # 计算并保存当前步骤的中间状态。
contaminated56[-1]["prompt"] = copy.deepcopy(DATA56[0]["prompt"])  # 计算并保存当前步骤的中间状态。
contaminated56[-1]["chosen"] = copy.deepcopy(DATA56[0]["rejected"])  # 计算并保存当前步骤的中间状态。
contaminated56[-1]["rejected"] = copy.deepcopy(DATA56[0]["chosen"])  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    validate_preference_data56(contaminated56, SPLIT56, TOKENIZER56)  # 执行当前语句以推进本节示例。
    raise AssertionError("reversed duplicate pair crossed splits")  # 遇到非法合同立即显式失败。
except ValueError as error56:  # 捕获预期异常并验证失败分支。
    assert str(error56) in {"prompt_cross_split_leakage", "pair_cross_split_leakage"}  # 用受控断言验证关键不变量。

duplicate56 = copy.deepcopy(DATA56)  # 计算并保存当前步骤的中间状态。
duplicate56.append({**copy.deepcopy(DATA56[0]), "id": "p7"})  # 执行当前语句以推进本节示例。
duplicate_split56 = copy.deepcopy(SPLIT56)  # 计算并保存当前步骤的中间状态。
duplicate_split56["train"].append("p7")  # 执行当前语句以推进本节示例。
try:  # 尝试执行可能失败的受控操作。
    validate_preference_data56(duplicate56, duplicate_split56, TOKENIZER56)  # 执行当前语句以推进本节示例。
    raise AssertionError("duplicate pair inside train was accepted")  # 遇到非法合同立即显式失败。
except ValueError as error56:  # 捕获预期异常并验证失败分支。
    assert str(error56) in {"duplicate_prompt_in_split", "duplicate_pair_in_split"}  # 用受控断言验证关键不变量。

assert set(fingerprints56["known_rule_regression"]) == set(fingerprints56["train"])  # 用受控断言验证关键不变量。
assert set(fingerprints56["true_holdout"]).isdisjoint(fingerprints56["train"])  # 用受控断言验证关键不变量。

## 7. 发布合同：policy、reference 与完整偏好数据一起绑定

可复现实验不能只存 policy。制品同时保存 policy/reference state，并在 key 前加命名空间；摘要逐项覆盖 key、dtype、shape、bytes。metadata 绑定完整 tokenizer、全部 pair、split、β、label smoothing、length_mode、EOS 语义和允许主体。

包内 self_digest 可由攻击者重签，所以 loader 必须把计算出的整体摘要与包外 TRUSTED_RELEASES56 比较。返回 PublishedDPOPolicy56，候选比较始终沿用发布时的 EOS 和 sum 语义。

In [ ]:
def state_digest56(state):  # 定义本节可复用的核心函数。
    digest = hashlib.sha256()  # 计算并保存当前步骤的中间状态。
    for key in sorted(state):  # 遍历输入元素以累积或检查结果。
        tensor = state[key].detach().cpu().contiguous()  # 计算并保存当前步骤的中间状态。
        header = canonical_json56({"key": key, "dtype": str(tensor.dtype), "shape": list(tensor.shape)}).encode()  # 计算并保存当前步骤的中间状态。
        raw = tensor.numpy().tobytes()  # 计算并保存当前步骤的中间状态。
        digest.update(len(header).to_bytes(8, "big")); digest.update(header)  # 执行当前语句以推进本节示例。
        digest.update(len(raw).to_bytes(8, "big")); digest.update(raw)  # 执行当前语句以推进本节示例。
    return digest.hexdigest()  # 返回当前分支计算出的结果。

CONFIG56 = {"vocab_size": len(TOKENIZER56), "dim": 20}  # 计算并保存当前步骤的中间状态。
RECIPE56 = {  # 计算并保存当前步骤的中间状态。
    "algorithm": "DPO", "beta": 0.3, "label_smoothing": 0.05,  # 执行当前语句以推进本节示例。
    "length_mode": "sum", "score_eos": True, "seed": SEED56,  # 执行当前语句以推进本节示例。
    "optimizer": "AdamW", "learning_rate": 0.045, "steps": 85,  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。

def flatten_states56(policy, reference):  # 定义本节可复用的核心函数。
    state = {}  # 计算并保存当前步骤的中间状态。
    for prefix, model in (("policy", policy), ("reference", reference)):  # 遍历输入元素以累积或检查结果。
        for key, value in model.state_dict().items():  # 遍历输入元素以累积或检查结果。
            state[prefix + "::" + key] = value.detach().cpu().clone()  # 计算并保存当前步骤的中间状态。
    return state  # 返回当前分支计算出的结果。

def release_digest56(package):  # 定义本节可复用的核心函数。
    envelope = {  # 计算并保存当前步骤的中间状态。
        "release_id": package["release_id"], "metadata": package["metadata"],  # 执行当前语句以推进本节示例。
        "state_digest": state_digest56(package["state"]),  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。
    return sha56(canonical_json56(envelope).encode())  # 返回当前分支计算出的结果。

def build_release56(policy, reference):  # 定义本节可复用的核心函数。
    state = flatten_states56(policy, reference)  # 计算并保存当前步骤的中间状态。
    package = {  # 计算并保存当前步骤的中间状态。
        "release_id": "dpo-preference-v1",  # 执行当前语句以推进本节示例。
        "metadata": {  # 执行当前语句以推进本节示例。
            "config": copy.deepcopy(CONFIG56), "tokenizer": copy.deepcopy(TOKENIZER56),  # 执行当前语句以推进本节示例。
            "data": copy.deepcopy(DATA56), "split": copy.deepcopy(SPLIT56),  # 执行当前语句以推进本节示例。
            "recipe": copy.deepcopy(RECIPE56), "allowed_subject": "alignment-eval",  # 执行当前语句以推进本节示例。
        },  # 执行当前语句以推进本节示例。
        "state": state, "state_digest": state_digest56(state),  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。
    package["self_digest"] = release_digest56(package)  # 计算并保存当前步骤的中间状态。
    return package  # 返回当前分支计算出的结果。

RELEASE_PACKAGE56 = build_release56(policy56, reference56)  # 计算并保存当前步骤的中间状态。
TRUSTED_RELEASES56 = MappingProxyType({"dpo-preference-v1": release_digest56(RELEASE_PACKAGE56)})  # 计算并保存当前步骤的中间状态。

class PublishedDPOPolicy56:  # 定义承载本节状态与行为的数据结构。
    def __init__(self, policy, metadata, subject):  # 定义本节可复用的核心函数。
        self._policy = policy.eval()  # 计算并保存当前步骤的中间状态。
        self._scorer = MaskedSequenceLogProb56(metadata["recipe"]["length_mode"])  # 计算并保存当前步骤的中间状态。
        self.tokenizer = MappingProxyType(copy.deepcopy(metadata["tokenizer"]))  # 计算并保存当前步骤的中间状态。
        self.subject = subject  # 计算并保存当前步骤的中间状态。

    @torch.no_grad()  # 为下方定义附加声明式配置。
    def choose(self, prompt, candidate_a, candidate_b):  # 定义本节可复用的核心函数。
        if self.subject != "alignment-eval":  # 按当前条件选择后续控制路径。
            raise PermissionError("subject_not_authorized")  # 遇到非法合同立即显式失败。
        if not isinstance(prompt, list) or len(prompt) < 2 or prompt[0] != self.tokenizer["<bos>"]:  # 按当前条件选择后续控制路径。
            raise ValueError("invalid_published_prompt")  # 遇到非法合同立即显式失败。
        allowed_candidates = {"CLEAR", "CAUTIOUS"}  # 计算并保存当前步骤的中间状态。
        if candidate_a not in allowed_candidates or candidate_b not in allowed_candidates or candidate_a == candidate_b:  # 按当前条件选择后续控制路径。
            raise ValueError("invalid_candidate_pair")  # 遇到非法合同立即显式失败。
        sequences = [  # 计算并保存当前步骤的中间状态。
            prompt + [self.tokenizer[candidate], self.tokenizer["<eos>"]]  # 执行当前语句以推进本节示例。
            for candidate in (candidate_a, candidate_b)  # 遍历输入元素以累积或检查结果。
        ]  # 执行当前语句以推进本节示例。
        ids = torch.tensor(sequences, dtype=torch.long)  # 计算并保存当前步骤的中间状态。
        attention = torch.ones_like(ids, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
        response = torch.zeros((2, ids.shape[1] - 1), dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
        response[:, len(prompt) - 1:] = True  # 计算并保存当前步骤的中间状态。
        scores = self._scorer(self._policy, ids, attention, response)  # 计算并保存当前步骤的中间状态。
        return (candidate_a, candidate_b)[int(scores[1] > scores[0])]  # 返回当前分支计算出的结果。

def load_published56(package, subject):  # 定义本节可复用的核心函数。
    release_id = package.get("release_id")  # 计算并保存当前步骤的中间状态。
    actual = release_digest56(package)  # 计算并保存当前步骤的中间状态。
    if release_id not in TRUSTED_RELEASES56 or actual != TRUSTED_RELEASES56[release_id]:  # 按当前条件选择后续控制路径。
        raise PermissionError("untrusted_release_digest")  # 遇到非法合同立即显式失败。
    if package.get("self_digest") != actual or package.get("state_digest") != state_digest56(package["state"]):  # 按当前条件选择后续控制路径。
        raise ValueError("corrupt_release")  # 遇到非法合同立即显式失败。
    metadata = package["metadata"]  # 计算并保存当前步骤的中间状态。
    validate_preference_data56(metadata["data"], metadata["split"], metadata["tokenizer"])  # 执行当前语句以推进本节示例。
    recipe = metadata["recipe"]  # 计算并保存当前步骤的中间状态。
    if recipe != RECIPE56 or metadata["config"]["vocab_size"] != len(metadata["tokenizer"]):  # 按当前条件选择后续控制路径。
        raise ValueError("training_semantics_mismatch")  # 遇到非法合同立即显式失败。
    if subject != metadata["allowed_subject"]:  # 按当前条件选择后续控制路径。
        raise PermissionError("subject_not_authorized")  # 遇到非法合同立即显式失败。
    policy = TinyAutoregressivePolicy56(**metadata["config"])  # 计算并保存当前步骤的中间状态。
    reference = TinyAutoregressivePolicy56(**metadata["config"])  # 计算并保存当前步骤的中间状态。
    policy_state = {key.split("::", 1)[1]: value for key, value in package["state"].items() if key.startswith("policy::")}  # 计算并保存当前步骤的中间状态。
    reference_state = {key.split("::", 1)[1]: value for key, value in package["state"].items() if key.startswith("reference::")}  # 计算并保存当前步骤的中间状态。
    policy.load_state_dict(policy_state, strict=True); reference.load_state_dict(reference_state, strict=True)  # 计算并保存当前步骤的中间状态。
    for parameter in reference.parameters():  # 遍历输入元素以累积或检查结果。
        parameter.requires_grad_(False)  # 执行当前语句以推进本节示例。
    return PublishedDPOPolicy56(policy, metadata, subject)  # 返回当前分支计算出的结果。

published56 = load_published56(copy.deepcopy(RELEASE_PACKAGE56), "alignment-eval")  # 计算并保存当前步骤的中间状态。
assert published56.choose([1, 9, 3], "CLEAR", "CAUTIOUS") == "CLEAR"  # 用受控断言验证关键不变量。
assert published56.choose([1, 10, 4], "CLEAR", "CAUTIOUS") == "CAUTIOUS"  # 用受控断言验证关键不变量。
assert isinstance(TRUSTED_RELEASES56, MappingProxyType)  # 用受控断言验证关键不变量。

## 8. 整体重签攻击与接口失败模式

攻击者修改 policy tensor 后可重算 state_digest 和 self_digest；若 loader 信任包内字段，伪造会成功。下列 oracle 确认包外 registry 拒绝这类整体重签。另测试错误主体、相同候选、非法 β 与空监督。

生产还要绑定 base/SFT checkpoint、聊天模板、截断侧、数据许可和 reference 部署精度；混合精度下 policy/reference 不同数值路径也可能改变 margin。

In [ ]:
forged56 = copy.deepcopy(RELEASE_PACKAGE56)  # 计算并保存当前步骤的中间状态。
forged_key56 = next(key for key in forged56["state"] if key.startswith("policy::"))  # 计算并保存当前步骤的中间状态。
forged56["state"][forged_key56].view(-1)[0] += 0.5  # 计算并保存当前步骤的中间状态。
forged56["state_digest"] = state_digest56(forged56["state"])  # 计算并保存当前步骤的中间状态。
forged56["self_digest"] = release_digest56(forged56)  # 计算并保存当前步骤的中间状态。
assert forged56["self_digest"] == release_digest56(forged56)  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    load_published56(forged56, "alignment-eval")  # 执行当前语句以推进本节示例。
    raise AssertionError("fully resigned forged policy was trusted")  # 遇到非法合同立即显式失败。
except PermissionError as error56:  # 捕获预期异常并验证失败分支。
    assert str(error56) == "untrusted_release_digest"  # 用受控断言验证关键不变量。

try:  # 尝试执行可能失败的受控操作。
    load_published56(RELEASE_PACKAGE56, "unknown")  # 执行当前语句以推进本节示例。
    raise AssertionError("unauthorized subject was accepted")  # 遇到非法合同立即显式失败。
except PermissionError as error56:  # 捕获预期异常并验证失败分支。
    assert str(error56) == "subject_not_authorized"  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    published56.choose([1, 5, 3], "CLEAR", "CLEAR")  # 执行当前语句以推进本节示例。
    raise AssertionError("identical candidates were accepted")  # 遇到非法合同立即显式失败。
except ValueError as error56:  # 捕获预期异常并验证失败分支。
    assert str(error56) == "invalid_candidate_pair"  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    DPOObjective56(beta=0)  # 计算并保存当前步骤的中间状态。
    raise AssertionError("zero beta was accepted")  # 遇到非法合同立即显式失败。
except ValueError as error56:  # 捕获预期异常并验证失败分支。
    assert str(error56) == "invalid_beta_or_label_smoothing"  # 用受控断言验证关键不变量。

## 9. 复杂度、边界与原始资料

DPO 不需要在线采样 reward model，但每个 pair 仍需 policy chosen/rejected 与 reference chosen/rejected 四条序列分数。缓存 reference log-prob 可省计算，却必须绑定 reference state、tokenizer、模板、截断和精度。长序列的主要成本仍来自底层自回归模型，本例的一阶模型只是算法显微镜。

原始资料：

- Direct Preference Optimization：https://arxiv.org/abs/2305.18290
- InstructGPT（偏好数据与 RLHF 背景）：https://arxiv.org/abs/2203.02155
- PyTorch logsigmoid 官方文档：https://pytorch.org/docs/stable/generated/torch.nn.functional.logsigmoid.html